In [ ]:
# Import required packages
import numpy as np
import pandas as pd

import requests

from scipy.stats import genextreme

import matplotlib.pyplot as plt

In [ ]:
# Mount google drive and check the directory
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
def url_download(url, fpath):
   response = requests.get(url)
   if response.status_code == 200:
      # Save the file locally
      with open(fpath, 'wb') as file:
          file.write(response.content)

### data from 1950 to 2024
url='https://apihub.kma.go.kr/api/typ01/url/kma_sfcdd3.php?tm1=####&tm2=####&stn=108&obs=TA&help=1&authKey=aIZ7k2xRTX6Ge5NsUZ1-IA'
fpath='/content/drive/MyDrive/Labs/2025BigData/kma_temp_long_108.csv'

url_download(url, fpath)

In [ ]:
df = pd.read_csv(fpath,
                 encoding='euc-kr',
                 comment='#', # Ignore comment lines starting with '#'
                 sep=r'\s+',  # To handle whitespace separation
                 header=None,
                 index_col=0,
                 parse_dates=True,
                 on_bad_lines='skip' # Skip lines with inconsistent number of columns)
                 )
### check your data

In [ ]:
### extract max value from each year
annual_max =

In [ ]:
# 1. GEV fitting (optimal parameter values)
fitted_shape, fitted_loc, fitted_scale = genextreme.fit(data)
print(f"Fitted: shape={fitted_shape:.3f}, loc={fitted_loc:.3f}, scale={fitted_scale:.3f}")

#
x = np.linspace(min(data) - 5, max(data) + 5, 1000)

#
gev_pdf = genextreme.pdf(x, fitted_shape, loc=fitted_loc, scale=fitted_scale)

### main plot
plt.figure(figsize=(8, 4))
#histogram
#gev pdf

plt.title("GEV Fitting vs Actual Data")
plt.xlabel("Value")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
def get_return_level(T, shape, loc, scale):
    return genextreme.ppf(1 - 1/T, shape, loc=loc, scale=scale)

# examples: return period of 10, 50, and 100 years
for T in [10, 50, 100]:
    z_T = get_return_level(T, fitted_shape, fitted_loc, fitted_scale)
    print(f"{T}-year return level: {z_T:.2f}")

In [ ]:
# 'shape', 'loc', 'scale'
vary_param = 'scale'

#
delta = 0.2  # ratio
if vary_param == 'shape':
    values = [fitted_shape - delta, fitted_shape, fitted_shape + delta]
elif vary_param == 'loc':
    values = [fitted_loc - delta * 20, fitted_loc, fitted_loc + delta * 20]
elif vary_param == 'scale':
    values = [fitted_scale * 0.8, fitted_scale, fitted_scale * 1.2]
else:
    raise ValueError("vary_param must be 'shape', 'loc', or 'scale'")

#
x = np.linspace(min(data) - 5, max(data) + 5, 1000)

# 5
plt.figure(figsize=(8, 4))
plt.hist(data, bins=15, density=True, alpha=0.5, label="Annual Max Temp (Data)")

# 6. 선택된 파라미터만 바꿔서 분포 비교
for val in values:
    if vary_param == 'shape':
        pdf = genextreme.pdf(x, val, loc=fitted_loc, scale=fitted_scale)
        loglik = np.sum(genextreme.logpdf(data, val, loc=fitted_loc, scale=fitted_scale))
        label = f"shape={val:.2f}, logL={loglik:.1f}"
    elif vary_param == 'loc':
        pdf = genextreme.pdf(x, fitted_shape, loc=val, scale=fitted_scale)
        loglik = np.sum(genextreme.logpdf(data, fitted_shape, loc=val, scale=fitted_scale))
        label = f"loc={val:.2f}, logL={loglik:.1f}"
    elif vary_param == 'scale':
        pdf = genextreme.pdf(x, fitted_shape, loc=fitted_loc, scale=val)
        loglik = np.sum(genextreme.logpdf(data, fitted_shape, loc=fitted_loc, scale=val))
        label = f"scale={val:.2f}, logL={loglik:.1f}"

    plt.plot(x, pdf, label=label)

#
plt.title(f"GEV Fitting: Varying '{vary_param}' Parameter")
plt.xlabel("Temperature (°C)")
plt.ylabel("Density")
plt.legend()
plt.grid(True)
plt.show()